# LLM Recommendation System — Full Preprocessing Pipeline 

Author: Jaswanthi Mandalapu

Student ID: 019148883


AI Disclosure: I acknowledge that ChatGPT assisted me in coding several modules of this project, while the core ideas were developed independently as a group

In [46]:
import os, json, math, random
import duckdb
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

# ── PATHS ──────────────────────────────────────────────────
BASE_DIR   = r"/Users/jaswanthimandalapu/Documents/Sem2/CMPE 256 (Recommender Systems)/Final Project"
RAW_FILE   = os.path.join(BASE_DIR, "HotelRec.txt")
OUTPUT_DIR = os.path.join(BASE_DIR, "llm_rec_data_20interactions")
DB_PATH    = os.path.join(BASE_DIR, "llm_rec_pipeline.duckdb")
TMP_DIR    = os.path.join(OUTPUT_DIR, "tmp")

# ── PIPELINE PARAMS ────────────────────────────────────────
K_CORE              = 20    # min interactions per user / item
MAX_PROFILE_REVIEWS = 10   # reviews included per hotel / user text profile
NEG_SAMPLES         = 100  # random negative items pre-sampled per user

for d in [OUTPUT_DIR, TMP_DIR,
          os.path.join(OUTPUT_DIR, "core"),
          os.path.join(OUTPUT_DIR, "item"),
          os.path.join(OUTPUT_DIR, "user"),
          os.path.join(OUTPUT_DIR, "training")]:
    os.makedirs(d, exist_ok=True)

RAW_SAFE = RAW_FILE.replace("\\", "/")
OUT_SAFE = OUTPUT_DIR.replace("\\", "/")

print("RAW FILE :", RAW_FILE)
print("OUTPUT   :", OUTPUT_DIR)
print("DB       :", DB_PATH)

RAW FILE : /Users/jaswanthimandalapu/Documents/Sem2/CMPE 256 (Recommender Systems)/Final Project/HotelRec.txt
OUTPUT   : /Users/jaswanthimandalapu/Documents/Sem2/CMPE 256 (Recommender Systems)/Final Project/llm_rec_data_20interactions
DB       : /Users/jaswanthimandalapu/Documents/Sem2/CMPE 256 (Recommender Systems)/Final Project/llm_rec_pipeline.duckdb


In [47]:
con = duckdb.connect(DB_PATH)
#con.execute(f"PRAGMA temp_directory = '{TMP_DIR.replace(chr(92), chr(47))}'")
con.execute("PRAGMA memory_limit = '8GB'")
# Checkpoint frequently so DuckDB reclaims space from dropped tables
con.execute("PRAGMA checkpoint_threshold = '500MB'")
print("DuckDB:", con.execute("SELECT version()").fetchone()[0])

DuckDB: v1.5.2


## Stage 1 — Load Raw Data

Read `HotelRec.txt` (newline-delimited JSON) into DuckDB.
Keep all fields: hotel URL, author, timestamp, rating, review title, review text, and all six sub-ratings
(`value`, `rooms`, `location`, `cleanliness`, `service`, `sleep quality`).
Sub-ratings of 0 are treated as missing (0 is not a valid TripAdvisor sub-rating).


In [48]:
# Load only the columns needed for filtering — text and sub-ratings are
# excluded here and joined back from the raw file AFTER k-core shrinks the
# dataset (50 M → ~10 M rows), keeping the .duckdb file small.
con.execute("DROP TABLE IF EXISTS raw_lean")

con.execute(f"""
CREATE TABLE raw_lean AS
SELECT
    hotel_url                       AS item_id,
    author                          AS user_id,
    TRY_CAST("date" AS TIMESTAMP)   AS ts,
    TRY_CAST(rating AS DOUBLE)      AS rating
FROM read_json_auto('{RAW_SAFE}', format='newline_delimited', sample_size=-1)
""")

n = con.execute("SELECT COUNT(*) FROM raw_lean").fetchone()[0]
print(f"Loaded {n:,} rows (lean — text and sub-ratings excluded until Stage 6.5)")

Loaded 50,264,531 rows (lean — text and sub-ratings excluded until Stage 6.5)


In [49]:
# Lean audit (text/sub-rating counts not available yet — they come back in Stage 6.5)
audit = con.execute("""
SELECT
    COUNT(*)                           AS n_rows,
    COUNT(DISTINCT user_id)            AS n_users,
    COUNT(DISTINCT item_id)            AS n_items,
    MIN(ts)                            AS min_date,
    MAX(ts)                            AS max_date,
    MIN(rating)                        AS min_rating,
    MAX(rating)                        AS max_rating
FROM raw_lean
""").fetchdf()
display(audit)

,n_rows,n_users,n_items,min_date,max_date,min_rating,max_rating
0,50264531,21891404,365057,2001-02-01,2019-09-20,1.0,5.0


## Stage 2 — Validity Filtering

Remove rows that cannot be used for modeling:
- Null, empty, or known placeholder user names
- Null or out-of-range ratings (must be 1–5)
- Missing timestamps or hotel URLs


In [50]:
PLACEHOLDER_USERS = (
    "A TripAdvisor Member", "Anonymous", "TripAdvisor Member",
    "A Trip Advisor Member", ""
)
placeholder_sql = ", ".join(f"'{u}'" for u in PLACEHOLDER_USERS)

con.execute("DROP TABLE IF EXISTS clean_lean")
con.execute(f"""
CREATE TABLE clean_lean AS
SELECT * FROM raw_lean
WHERE user_id IS NOT NULL
  AND TRIM(user_id) != ''
  AND user_id NOT IN ({placeholder_sql})
  AND item_id IS NOT NULL
  AND rating IS NOT NULL
  AND rating BETWEEN 1 AND 5
  AND ts IS NOT NULL
""")

n_raw   = con.execute("SELECT COUNT(*) FROM raw_lean").fetchone()[0]
n_clean = con.execute("SELECT COUNT(*) FROM clean_lean").fetchone()[0]
print(f"Before: {n_raw:,}  After: {n_clean:,}  Removed: {n_raw - n_clean:,}")

# Free space immediately
con.execute("DROP TABLE raw_lean")
con.execute("CHECKPOINT")
print("raw_lean dropped and checkpointed.")

Before: 50,264,531  After: 50,264,530  Removed: 1
raw_lean dropped and checkpointed.


## Stage 3 — Deduplication

Some users reviewed the same hotel more than once. For modeling we keep only the **most recent**
interaction per (user, hotel) pair (tiebreak: higher rating). The full duplicate history is
saved separately as `duplicate_features.parquet` — useful as a repeat-visit feature for
LLM-based models that can condition on user loyalty.


In [51]:
# dup_features only needs (user_id, item_id, rating, ts) — lean data is sufficient
con.execute("DROP TABLE IF EXISTS dup_features")
con.execute("""
CREATE TABLE dup_features AS
WITH pair_stats AS (
    SELECT
        user_id, item_id,
        COUNT(*)           AS visit_count,
        MIN(rating)        AS min_rating,
        MAX(rating)        AS max_rating,
        MAX(rating) - MIN(rating) AS rating_change,
        MIN(ts)            AS first_visit,
        MAX(ts)            AS last_visit
    FROM clean_lean
    GROUP BY user_id, item_id
    HAVING COUNT(*) > 1
)
SELECT *,
    DATEDIFF('day', first_visit, last_visit) AS days_between_visits
FROM pair_stats
ORDER BY visit_count DESC
""")

n_dup_pairs = con.execute("SELECT COUNT(*) FROM dup_features").fetchone()[0]
print(f"Duplicate (user, hotel) pairs: {n_dup_pairs:,}")

# Keep only the most recent interaction per (user, item) pair
con.execute("DROP TABLE IF EXISTS dedup_lean")
con.execute("""
CREATE TABLE dedup_lean AS
WITH ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY user_id, item_id
            ORDER BY ts DESC, rating DESC
        ) AS rn
    FROM clean_lean
)
SELECT * EXCLUDE (rn) FROM ranked WHERE rn = 1
""")

n_dedup = con.execute("SELECT COUNT(*) FROM dedup_lean").fetchone()[0]
print(f"After dedup: {n_dedup:,} interactions")
#con.execute(f"COPY dedup_lean TO '{OUT_SAFE}/tmp/dedup_lean.parquet' (FORMAT PARQUET)")

# Free space immediately
con.execute("DROP TABLE clean_lean")
con.execute("CHECKPOINT")
print("clean_lean dropped and checkpointed.")

Duplicate (user, hotel) pairs: 1,068,974
After dedup: 47,793,515 interactions
clean_lean dropped and checkpointed.


## Stage 4 — 25-Core Filtering

Iteratively remove users with fewer than 5 interactions and hotels with fewer than 5 reviews
until the dataset is stable. This ensures every user and hotel has enough training signal
for any model — including LLMs that need a meaningful interaction history to condition on.


In [52]:
con.execute("DROP TABLE IF EXISTS kcore_current")
con.execute("CREATE TABLE kcore_current AS SELECT * FROM dedup_lean")
con.execute("DROP TABLE dedup_lean")
con.execute("CHECKPOINT")

iteration = 0
while True:
    iteration += 1

    con.execute("DROP TABLE IF EXISTS kcore_next")
    con.execute(f"""
    CREATE TABLE kcore_next AS
    SELECT * FROM kcore_current
    WHERE user_id IN (
        SELECT user_id FROM kcore_current
        GROUP BY user_id HAVING COUNT(*) >= {K_CORE}
    )
    AND item_id IN (
        SELECT item_id FROM kcore_current
        GROUP BY item_id HAVING COUNT(*) >= {K_CORE}
    )
    """)

    n_before = con.execute("SELECT COUNT(*) FROM kcore_current").fetchone()[0]
    n_after  = con.execute("SELECT COUNT(*) FROM kcore_next").fetchone()[0]
    min_u = con.execute("SELECT MIN(cnt) FROM (SELECT user_id, COUNT(*) AS cnt FROM kcore_next GROUP BY user_id)").fetchone()[0]
    min_i = con.execute("SELECT MIN(cnt) FROM (SELECT item_id, COUNT(*) AS cnt FROM kcore_next GROUP BY item_id)").fetchone()[0]

    print(f"Iter {iteration}: {n_before:,} → {n_after:,} rows | min_user={min_u}  min_item={min_i}")

    con.execute("DROP TABLE kcore_current")
    con.execute("ALTER TABLE kcore_next RENAME TO kcore_current")
    con.execute("CHECKPOINT")  # reclaim space from dropped table each iteration

    if n_after == n_before:
        print("Stable — 20-core complete")
        break

con.execute("DROP TABLE IF EXISTS kcore_final")
con.execute("ALTER TABLE kcore_current RENAME TO kcore_final")
con.execute("CHECKPOINT")

n_u = con.execute("SELECT COUNT(DISTINCT user_id) FROM kcore_final").fetchone()[0]
n_i = con.execute("SELECT COUNT(DISTINCT item_id) FROM kcore_final").fetchone()[0]
n_r = con.execute("SELECT COUNT(*) FROM kcore_final").fetchone()[0]
print(f"\n20-core dataset: {n_r:,} interactions | {n_u:,} users | {n_i:,} items")

Iter 1: 47,793,515 → 5,439,178 rows | min_user=3  min_item=1
Iter 2: 5,439,178 → 4,072,859 rows | min_user=1  min_item=14
Iter 3: 4,072,859 → 3,018,993 rows | min_user=14  min_item=1
Iter 4: 3,018,993 → 2,681,595 rows | min_user=4  min_item=15
Iter 5: 2,681,595 → 2,324,069 rows | min_user=17  min_item=4
Iter 6: 2,324,069 → 2,177,926 rows | min_user=8  min_item=17
Iter 7: 2,177,926 → 2,000,889 rows | min_user=17  min_item=8
Iter 8: 2,000,889 → 1,922,044 rows | min_user=11  min_item=17
Iter 9: 1,922,044 → 1,820,734 rows | min_user=18  min_item=12
Iter 10: 1,820,734 → 1,771,369 rows | min_user=13  min_item=18
Iter 11: 1,771,369 → 1,706,464 rows | min_user=18  min_item=14
Iter 12: 1,706,464 → 1,674,450 rows | min_user=15  min_item=18
Iter 13: 1,674,450 → 1,631,442 rows | min_user=18  min_item=14
Iter 14: 1,631,442 → 1,608,774 rows | min_user=15  min_item=18
Iter 15: 1,608,774 → 1,577,314 rows | min_user=18  min_item=16
Iter 16: 1,577,314 → 1,560,989 rows | min_user=15  min_item=17
Iter 17:

## Stage 5 — Leave-Last-Out Temporal Split

For each user, rank interactions by timestamp (most recent first):
- **test** → most recent interaction (rank 1)
- **valid** → second most recent (rank 2)
- **train** → everything else (rank ≥ 3)

This is the standard evaluation protocol for sequential/LLM recommendation.
The model sees the user's full history up to some point and must predict the next item.


In [53]:
con.execute("DROP TABLE IF EXISTS split_lean")
con.execute("""
CREATE TABLE split_lean AS
WITH ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY ts DESC, rating DESC
        ) AS rn_desc
    FROM kcore_final
)
SELECT
    ROW_NUMBER() OVER (ORDER BY user_id, ts) AS interaction_id,
    user_id, item_id, ts AS timestamp, rating,
    CASE
        WHEN rn_desc = 1 THEN 'test'
        WHEN rn_desc = 2 THEN 'valid'
        ELSE 'train'
    END AS split
FROM ranked
""")

split_summary = con.execute("""
SELECT split,
    COUNT(*) AS n_interactions,
    COUNT(DISTINCT user_id) AS n_users,
    COUNT(DISTINCT item_id) AS n_items,
    AVG(rating) AS mean_rating
FROM split_lean
GROUP BY split
ORDER BY split DESC
""").fetchdf()
display(split_summary)

con.execute("DROP TABLE kcore_final")
con.execute("CHECKPOINT")
print("kcore_final dropped and checkpointed.")

,split,n_interactions,n_users,n_items,mean_rating
0,valid,46660,46660,18731,4.143892
1,train,1306782,46660,27197,4.044277
2,test,46660,46660,18355,4.221496


kcore_final dropped and checkpointed.


## Stage 6 — Integer ID Mapping

Map string user IDs and hotel URLs to contiguous 0-based integers.
Required for embedding lookups, sparse matrix operations, and sequence indexing in LLM models.


In [54]:
con.execute("DROP TABLE IF EXISTS user_map")
con.execute("""
CREATE TABLE user_map AS
SELECT user_id, (ROW_NUMBER() OVER (ORDER BY user_id) - 1) AS user_idx
FROM (SELECT DISTINCT user_id FROM split_lean)
""")

con.execute("DROP TABLE IF EXISTS item_map")
con.execute("""
CREATE TABLE item_map AS
SELECT item_id, (ROW_NUMBER() OVER (ORDER BY item_id) - 1) AS item_idx
FROM (SELECT DISTINCT item_id FROM split_lean)
""")

# final_lean: all filtered interactions with integer IDs, but still no text
con.execute("DROP TABLE IF EXISTS final_lean")
con.execute("""
CREATE TABLE final_lean AS
SELECT
    s.interaction_id,
    u.user_idx, s.user_id,
    i.item_idx, s.item_id,
    s.timestamp, s.rating, s.split
FROM split_lean s
JOIN user_map u USING (user_id)
JOIN item_map i USING (item_id)
""")

n_users = con.execute("SELECT COUNT(*) FROM user_map").fetchone()[0]
n_items = con.execute("SELECT COUNT(*) FROM item_map").fetchone()[0]
n_total = con.execute("SELECT COUNT(*) FROM final_lean").fetchone()[0]
print(f"Users: {n_users:,}  Items: {n_items:,}  Total interactions: {n_total:,}")

con.execute("DROP TABLE split_lean")
con.execute("CHECKPOINT")
print("split_lean dropped. DB now contains only lean filtered data.")

Users: 46,660  Items: 27,197  Total interactions: 1,400,102
split_lean dropped. DB now contains only lean filtered data.


## Stage 6.5 — Content Join

Now that k-core has reduced the dataset from 50 M to ~10 M rows, re-scan the
raw JSON **once** and join on `(user_id, item_id, timestamp, rating)` to attach
review text and sub-ratings to only the surviving interactions.

This keeps the `.duckdb` file small during all filtering stages and only
materialises text for the ~10 M rows that actually matter.

In [55]:
print("Re-scanning raw file to attach text/sub-ratings to filtered interactions...")
print(f"Joining {con.execute('SELECT COUNT(*) FROM final_lean').fetchone()[0]:,} rows with raw JSON...")

con.execute("DROP TABLE IF EXISTS final_data")
con.execute(f"""
CREATE TABLE final_data AS
SELECT
    fl.interaction_id,
    fl.user_idx,  fl.user_id,
    fl.item_idx,  fl.item_id,
    fl.timestamp, fl.rating,  fl.split,
    NULLIF(TRIM(raw.title),  '') AS title,
    NULLIF(TRIM(raw."text"), '') AS review_text,
    NULLIF(TRY_CAST(json_extract_string(raw.property_dict, '$.value')          AS DOUBLE), 0) AS sub_value,
    NULLIF(TRY_CAST(json_extract_string(raw.property_dict, '$.rooms')          AS DOUBLE), 0) AS sub_rooms,
    NULLIF(TRY_CAST(json_extract_string(raw.property_dict, '$.location')       AS DOUBLE), 0) AS sub_location,
    NULLIF(TRY_CAST(json_extract_string(raw.property_dict, '$.cleanliness')    AS DOUBLE), 0) AS sub_cleanliness,
    NULLIF(TRY_CAST(json_extract_string(raw.property_dict, '$.service')        AS DOUBLE), 0) AS sub_service,
    NULLIF(TRY_CAST(json_extract_string(raw.property_dict, '$.sleep quality')  AS DOUBLE), 0) AS sub_sleep_quality
FROM final_lean fl
JOIN (
    SELECT
        hotel_url, author,
        TRY_CAST("date" AS TIMESTAMP) AS ts,
        TRY_CAST(rating AS DOUBLE)    AS rtg,
        title, "text", property_dict,
        -- deduplicate raw rows that share the same join key
        ROW_NUMBER() OVER (
            PARTITION BY hotel_url, author,
                         TRY_CAST("date" AS TIMESTAMP),
                         TRY_CAST(rating AS DOUBLE)
            ORDER BY (SELECT NULL)
        ) AS raw_rn
    FROM read_json_auto('{RAW_SAFE}', format='newline_delimited', sample_size=-1)
) raw
    ON  raw.hotel_url = fl.item_id
    AND raw.author    = fl.user_id
    AND raw.ts        = fl.timestamp
    AND raw.rtg       = fl.rating
    AND raw.raw_rn    = 1
""")

n_joined = con.execute("SELECT COUNT(*) FROM final_data").fetchone()[0]
n_lean   = con.execute("SELECT COUNT(*) FROM final_lean").fetchone()[0]
print(f"final_data rows: {n_joined:,}  (expected {n_lean:,})")
if n_joined != n_lean:
    print(f"  WARNING: {abs(n_lean - n_joined):,} rows unmatched — check for timestamp precision issues")

con.execute("DROP TABLE final_lean")
con.execute("CHECKPOINT")
print("Content join complete. final_data (with text) ready for export stages.")

Re-scanning raw file to attach text/sub-ratings to filtered interactions...
Joining 1,400,102 rows with raw JSON...
final_data rows: 1,400,102  (expected 1,400,102)
Content join complete. final_data (with text) ready for export stages.


## Stage 6.6 — Subset to Top-2000 Most Active Users

To keep model training fast, we restrict the dataset to the **2,000 users with the most total
reviews** (across all splits). This preserves users who have the richest interaction histories,
which is ideal for evaluating sequential / LLM recommendation models.

Steps:
1. Rank all users by total review count and keep the top 2,000.
2. Filter `final_data` to those users (train / valid / test rows all trimmed).
3. Re-index `user_idx` and `item_idx` to be contiguous 0-based integers over the subset.
4. Redirect downstream output to `llm_rec_data_top2000/` (full dataset in `llm_rec_data/` is untouched).

In [8]:
TOP_N_USERS = 2000

# Identify top-N users by total review count across all splits
con.execute("DROP TABLE IF EXISTS top_users_list")
con.execute(f"""
CREATE TABLE top_users_list AS
SELECT user_id
FROM (
    SELECT user_id, COUNT(*) AS total_reviews
    FROM final_data
    GROUP BY user_id
    ORDER BY total_reviews DESC
    LIMIT {TOP_N_USERS}
)
""")

review_counts = con.execute("""
SELECT COUNT(*) AS total_reviews
FROM final_data
WHERE user_id IN (SELECT user_id FROM top_users_list)
GROUP BY user_id
ORDER BY total_reviews DESC
""").fetchdf()["total_reviews"]
print(f"Top-{TOP_N_USERS} user review counts:  min={review_counts.min()}  max={review_counts.max()}  mean={review_counts.mean():.1f}")

# Filter final_data to the top-N users and re-index IDs from 0
con.execute("DROP TABLE IF EXISTS final_data_full")
con.execute("ALTER TABLE final_data RENAME TO final_data_full")

con.execute("DROP TABLE IF EXISTS final_data")
con.execute("""
CREATE TABLE final_data AS
SELECT * FROM final_data_full
WHERE user_id IN (SELECT user_id FROM top_users_list)
""")

con.execute("DROP TABLE final_data_full")
con.execute("CHECKPOINT")
print("Filtered final_data to top users.")

# Re-build user_map and item_map for the subset (contiguous 0-based integers)
con.execute("DROP TABLE IF EXISTS user_map")
con.execute("""
CREATE TABLE user_map AS
SELECT user_id, (ROW_NUMBER() OVER (ORDER BY user_id) - 1) AS user_idx
FROM (SELECT DISTINCT user_id FROM final_data)
""")

con.execute("DROP TABLE IF EXISTS item_map")
con.execute("""
CREATE TABLE item_map AS
SELECT item_id, (ROW_NUMBER() OVER (ORDER BY item_id) - 1) AS item_idx
FROM (SELECT DISTINCT item_id FROM final_data)
""")

# Re-assign integer IDs in final_data using the new maps
con.execute("DROP TABLE IF EXISTS final_data_reindexed")
con.execute("""
CREATE TABLE final_data_reindexed AS
SELECT
    s.interaction_id,
    u.user_idx, s.user_id,
    i.item_idx, s.item_id,
    s.timestamp, s.rating, s.split,
    s.title, s.review_text,
    s.sub_value, s.sub_rooms, s.sub_location,
    s.sub_cleanliness, s.sub_service, s.sub_sleep_quality
FROM final_data s
JOIN user_map u USING (user_id)
JOIN item_map i USING (item_id)
""")

con.execute("DROP TABLE final_data")
con.execute("ALTER TABLE final_data_reindexed RENAME TO final_data")
con.execute("CHECKPOINT")

# Redirect all downstream outputs to a separate directory so the full dataset is preserved
OUTPUT_DIR = os.path.join(BASE_DIR, f"llm_rec_data_top{TOP_N_USERS}")
OUT_SAFE   = OUTPUT_DIR.replace("\\", "/")

for d in [OUTPUT_DIR,
          os.path.join(OUTPUT_DIR, "core"),
          os.path.join(OUTPUT_DIR, "item"),
          os.path.join(OUTPUT_DIR, "user"),
          os.path.join(OUTPUT_DIR, "training")]:
    os.makedirs(d, exist_ok=True)

print(f"Output directory set to: {OUTPUT_DIR}")

# Summary of the subset
subset_summary = con.execute("""
SELECT split,
    COUNT(*)                AS n_interactions,
    COUNT(DISTINCT user_id) AS n_users,
    COUNT(DISTINCT item_id) AS n_items,
    AVG(rating)             AS mean_rating
FROM final_data
GROUP BY split
ORDER BY split DESC
""").fetchdf()
print(f"\nTop-{TOP_N_USERS}-user subset:")
display(subset_summary)

Top-2000 user review counts:  min=94  max=38095  mean=152.8
Filtered final_data to top users.
Output directory set to: /Users/jaswanthimandalapu/Documents/Sem2/CMPE 256 (Recommender Systems)/Final Project/llm_rec_data_top2000

Top-2000-user subset:


,split,n_interactions,n_users,n_items,mean_rating
0,valid,2000,2000,1973,4.166500
1,train,301608,2000,129823,3.928238
2,test,2000,2000,1972,4.231000


## Stage 7 — Export Core Interactions

Export one parquet per split, each row containing the full interaction record:
integer IDs, string IDs, rating, timestamp, review title, review text, and all sub-ratings.
The review text travels with each interaction so downstream LLM training code
doesn't need to re-join against the raw file.


In [56]:
for split in ["train", "valid", "test"]:
    path = f"{OUT_SAFE}/core/interactions_{split}.parquet"
    con.execute(f"""
    COPY (
        SELECT interaction_id, user_idx, item_idx, user_id, item_id,
               timestamp, rating, split,
               title, review_text,
               sub_value, sub_rooms, sub_location,
               sub_cleanliness, sub_service, sub_sleep_quality
        FROM final_data
        WHERE split = '{split}'
        ORDER BY timestamp ASC, interaction_id ASC
    ) TO '{path}' (FORMAT PARQUET)
    """)
    n = con.execute(f"SELECT COUNT(*) FROM final_data WHERE split='{split}'").fetchone()[0]
    size_mb = os.path.getsize(path.replace("/", os.sep)) / 1e6
    print(f"  {split:6s}: {n:>10,} rows  →  {size_mb:.1f} MB")

# ID mapping tables
con.execute(f"COPY user_map TO '{OUT_SAFE}/core/user_id_mapping.parquet' (FORMAT PARQUET)")
con.execute(f"COPY item_map TO '{OUT_SAFE}/core/item_id_mapping.parquet' (FORMAT PARQUET)")

# Duplicate features with integer IDs joined in
con.execute(f"""
COPY (
    SELECT d.user_id, d.item_id, u.user_idx, i.item_idx,
           d.visit_count, d.min_rating, d.max_rating,
           d.rating_change, d.days_between_visits,
           d.first_visit, d.last_visit
    FROM dup_features d
    LEFT JOIN user_map u USING (user_id)
    LEFT JOIN item_map i USING (item_id)
    ORDER BY d.visit_count DESC
) TO '{OUT_SAFE}/core/duplicate_features.parquet' (FORMAT PARQUET)
""")

print("\ncore/ exports complete.")

  train :  1,306,782 rows  →  744.4 MB
  valid :     46,660 rows  →  26.9 MB
  test  :     46,660 rows  →  27.1 MB

core/ exports complete.


## Stage 8 — Item Data

Three outputs for items:
1. **`item_metadata.parquet`** — structured stats: hotel name (parsed from URL), location,
   geo ID, TripAdvisor hotel ID, review count, mean rating, Bayesian-smoothed popularity,
   bias term, and per-category sub-rating averages (value, rooms, location, cleanliness,
   service, sleep quality). All computed from **training data only**.
2. **`item_profiles.parquet`** — aggregated text: the most recent `MAX_PROFILE_REVIEWS`
   training reviews per hotel concatenated into a single document. This is the hotel's
   natural-language description usable as context in LLM prompts.


In [57]:
# Hotel name + location parsed from the TripAdvisor URL format:
# Hotel_Review-g{geo_id}-d{ta_id}-Reviews-{Hotel_Name_Parts}-{Location_Parts}.html
# e.g. Hotel_Review-g274707-d658737-Reviews-Hotel_Residence_Agnes-Prague_Bohemia.html
# hotel_name = "Hotel Residence Agnes",  location_raw = "Prague Bohemia"

con.execute("DROP TABLE IF EXISTS item_metadata")
con.execute(f"""
CREATE TABLE item_metadata AS
WITH
item_names AS (
    SELECT
        item_id,
        REGEXP_EXTRACT(item_id, '-g([0-9]+)-', 1)    AS geo_id,
        REGEXP_EXTRACT(item_id, '-d([0-9]+)-', 1)    AS ta_hotel_id,
        -- everything between "Reviews-" and ".html" is the slug
        SPLIT_PART(SPLIT_PART(item_id, 'Reviews-', 2), '.html', 1) AS slug
    FROM (SELECT DISTINCT item_id FROM final_data)
),
item_names2 AS (
    SELECT
        item_id, geo_id, ta_hotel_id,
        -- hotel name: first dash-separated component of the slug
        REPLACE(SPLIT_PART(slug, '-', 1), '_', ' ') AS hotel_name,
        -- location: rest of the slug after the first dash-separated component
        REPLACE(
            CASE WHEN slug LIKE '%-%'
                 THEN SUBSTR(slug, LENGTH(SPLIT_PART(slug, '-', 1)) + 2)
                 ELSE ''
            END,
        '_', ' ') AS location_raw
    FROM item_names
),
train_stats AS (
    SELECT
        item_idx, item_id,
        COUNT(*)           AS n_train_reviews,
        AVG(rating)        AS mean_rating,
        STDDEV(rating)     AS std_rating,
        AVG(sub_value)         AS avg_sub_value,
        AVG(sub_rooms)         AS avg_sub_rooms,
        AVG(sub_location)      AS avg_sub_location,
        AVG(sub_cleanliness)   AS avg_sub_cleanliness,
        AVG(sub_service)       AS avg_sub_service,
        AVG(sub_sleep_quality) AS avg_sub_sleep_quality,
        COUNT(review_text)     AS n_with_review_text
    FROM final_data
    WHERE split = 'train'
    GROUP BY item_idx, item_id
)
SELECT
    t.item_idx, t.item_id,
    n.hotel_name, n.location_raw, n.geo_id, n.ta_hotel_id,
    t.n_train_reviews, t.mean_rating, t.std_rating,
    t.avg_sub_value, t.avg_sub_rooms, t.avg_sub_location,
    t.avg_sub_cleanliness, t.avg_sub_service, t.avg_sub_sleep_quality,
    t.n_with_review_text
FROM train_stats t
JOIN item_names2 n USING (item_id)
ORDER BY t.item_idx
""")

# Bayesian-smoothed popularity score (train only)
# score = (v * R + m * C) / (v + m)
# v = review count, R = item mean, m = smoothing constant, C = global mean
global_mean = con.execute("SELECT AVG(rating) FROM final_data WHERE split='train'").fetchone()[0]
m_smooth    = int(con.execute("SELECT MEDIAN(n_train_reviews) FROM item_metadata").fetchone()[0])

con.execute("ALTER TABLE item_metadata ADD COLUMN bayesian_avg DOUBLE")
con.execute("ALTER TABLE item_metadata ADD COLUMN item_bias    DOUBLE")
con.execute(f"""
UPDATE item_metadata
SET
    bayesian_avg = (n_train_reviews * mean_rating + {m_smooth} * {global_mean})
                  / (n_train_reviews + {m_smooth}),
    item_bias    = mean_rating - {global_mean}
""")

print(f"Global train mean: {global_mean:.4f}  |  Smoothing m: {m_smooth}")
display(con.execute("SELECT * FROM item_metadata LIMIT 3").fetchdf())

Global train mean: 4.0443  |  Smoothing m: 32


,item_idx,item_id,hotel_name,location_raw,geo_id,ta_hotel_id,n_train_reviews,mean_rating,std_rating,avg_sub_value,avg_sub_rooms,avg_sub_location,avg_sub_cleanliness,avg_sub_service,avg_sub_sleep_quality,n_with_review_text,bayesian_avg,item_bias
0,0,Hotel_Review-g10006284-d1083311-Reviews-The_Re...,The Regent Grand,Grace Bay Providenciales Turks and Caicos,10006284,1083311,20,4.450000,0.686333,4.142857,4.777778,5.000000,4.875000,4.545455,4.857143,20,4.200324,0.405723
1,1,Hotel_Review-g10006284-d151184-Reviews-Club_Me...,Club Med Turkoise Turks Caicos,Grace Bay Providenciales Turks and Caicos,10006284,151184,57,3.701754,1.051613,3.833333,2.878788,4.764706,3.769231,3.782609,3.241379,57,3.824908,-0.342522
2,2,Hotel_Review-g10006284-d151225-Reviews-Ports_o...,Ports of Call Resort,Grace Bay Providenciales Turks and Caicos,10006284,151225,25,3.760000,1.011599,3.846154,3.375000,4.066667,3.666667,3.833333,4.000000,25,3.919594,-0.284277


In [58]:
# Build hotel text profiles from training reviews (capped at MAX_PROFILE_REVIEWS)
# Format: title (bold) + review text, separated by dividers, newest first

con.execute("DROP TABLE IF EXISTS item_profiles")
con.execute(f"""
CREATE TABLE item_profiles AS
WITH train_reviews AS (
    SELECT
        item_idx, item_id,
        title, review_text, timestamp, rating,
        ROW_NUMBER() OVER (
            PARTITION BY item_id
            ORDER BY timestamp DESC
        ) AS rn
    FROM final_data
    WHERE split = 'train'
      AND review_text IS NOT NULL
      AND LENGTH(TRIM(review_text)) > 20
),
top_reviews AS (
    SELECT * FROM train_reviews WHERE rn <= {MAX_PROFILE_REVIEWS}
)
SELECT
    item_idx,
    item_id,
    COUNT(*)  AS n_profile_reviews,
    STRING_AGG(
        CASE
            WHEN title IS NOT NULL
            THEN '[Rating: ' || CAST(CAST(rating AS INTEGER) AS VARCHAR) || '/5] '
                 || title || '\n' || review_text
            ELSE '[Rating: ' || CAST(CAST(rating AS INTEGER) AS VARCHAR) || '/5] '
                 || review_text
        END,
        '\n\n---\n\n'
        ORDER BY timestamp DESC
    ) AS aggregated_review_text
FROM top_reviews
GROUP BY item_idx, item_id
ORDER BY item_idx
""")

n_with_profiles = con.execute("SELECT COUNT(*) FROM item_profiles").fetchone()[0]
print(f"Item profiles built: {n_with_profiles:,} hotels have training review text")
display(con.execute("SELECT item_idx, item_id, n_profile_reviews, LEFT(aggregated_review_text, 200) AS preview FROM item_profiles LIMIT 3").fetchdf())

Item profiles built: 27,197 hotels have training review text


,item_idx,item_id,n_profile_reviews,preview
0,0,Hotel_Review-g10006284-d1083311-Reviews-The_Re...,10,[Rating: 5/5] Wonderful Location and Staff!\nO...
1,1,Hotel_Review-g10006284-d151184-Reviews-Club_Me...,10,[Rating: 4/5] Life’s a beach\nThere is a clue ...
2,2,Hotel_Review-g10006284-d151225-Reviews-Ports_o...,10,[Rating: 4/5] Nice surprise\nThe hotel where w...


In [59]:
con.execute(f"COPY item_metadata TO '{OUT_SAFE}/item/item_metadata.parquet' (FORMAT PARQUET)")
con.execute(f"COPY item_profiles TO '{OUT_SAFE}/item/item_profiles.parquet' (FORMAT PARQUET)")
print("item/ exports complete.")

item/ exports complete.


## Stage 9 — User Data

Three outputs for users:
1. **`user_metadata.parquet`** — review count, mean rating, user bias, activity span.
2. **`user_sequences.parquet`** — each user's training interactions in chronological order,
   flat format (one row per step), including the hotel name and review text at each step.
   This is the primary input for sequential LLM models (e.g., prompt: 'Given this user
   visited [hotel A, hotel B, ...], what hotel should we recommend next?').
3. **`user_profiles.parquet`** — each user's most recent training reviews concatenated into
   a single text document, structured as `[Hotel Name — Rating] review text`. Used as
   the user's natural-language preference context in LLM prompts.


In [60]:
# User metadata + bias terms (train only)
con.execute("DROP TABLE IF EXISTS user_metadata")
con.execute(f"""
CREATE TABLE user_metadata AS
WITH train_stats AS (
    SELECT
        user_idx, user_id,
        COUNT(*)          AS n_train_reviews,
        AVG(rating)       AS mean_rating,
        STDDEV(rating)    AS std_rating,
        MIN(timestamp)    AS first_review_date,
        MAX(timestamp)    AS last_review_date,
        COUNT(review_text)AS n_with_review_text
    FROM final_data
    WHERE split = 'train'
    GROUP BY user_idx, user_id
)
SELECT
    t.*,
    t.mean_rating - {global_mean}    AS user_bias,
    DATEDIFF('day', t.first_review_date, t.last_review_date) AS active_days
FROM train_stats t
ORDER BY t.user_idx
""")

display(con.execute("SELECT * FROM user_metadata LIMIT 3").fetchdf())

,user_idx,user_id,n_train_reviews,mean_rating,std_rating,first_review_date,last_review_date,n_with_review_text,user_bias,active_days
0,0,/undefined,9871,4.237463,0.981252,2001-06-01,2019-05-13 15:57:36,9871,0.193187,6555
1,1,001-Mrs-B-D,21,4.238095,0.624881,2014-12-01,2018-05-01 00:00:00,21,0.193819,1247
2,2,007BigBlue,43,4.232558,0.684426,2015-05-01,2018-04-01 00:00:00,43,0.188281,1066


In [61]:
# User sequences: ordered training history per user, flat format
# seq_position = 1 for the user's first-ever train review, ascending
# Joined with item_metadata to include hotel_name at each step

con.execute("DROP TABLE IF EXISTS user_sequences")
con.execute("""
CREATE TABLE user_sequences AS
SELECT
    f.user_idx, f.user_id,
    f.item_idx, f.item_id,
    m.hotel_name,
    f.timestamp, f.rating,
    f.title, f.review_text,
    f.sub_value, f.sub_rooms, f.sub_location,
    f.sub_cleanliness, f.sub_service, f.sub_sleep_quality,
    ROW_NUMBER() OVER (
        PARTITION BY f.user_id
        ORDER BY f.timestamp ASC, f.interaction_id ASC
    ) AS seq_position,
    COUNT(*) OVER (PARTITION BY f.user_id) AS train_seq_length
FROM final_data f
LEFT JOIN item_metadata m USING (item_idx)
WHERE f.split = 'train'
ORDER BY f.user_idx, seq_position
""")

n_seq = con.execute("SELECT COUNT(*) FROM user_sequences").fetchone()[0]
print(f"User sequence rows: {n_seq:,}")
display(con.execute("SELECT user_idx, user_id, seq_position, hotel_name, rating, LEFT(review_text, 100) AS review_preview FROM user_sequences WHERE user_idx=0 LIMIT 5").fetchdf())

User sequence rows: 1,306,782


,user_idx,user_id,seq_position,hotel_name,rating,review_preview
0,0,/undefined,1,Gunpowder House Suites,4.0,"Charming place. 9 Clean rooms, some with AC. S..."
1,0,/undefined,2,Admiral s Inn Gunpowder Suites,4.0,"Charming place. 9 Clean rooms, some with AC. S..."
2,0,/undefined,3,Chateau de l Argoat,4.0,I have been staying here off and on for about ...
3,0,/undefined,4,Inn by the Sea,5.0,Lovely property with casual elegance...balconi...
4,0,/undefined,5,Ambassador Hotel Tulsa Autograph Collection,5.0,The Ambassador in Tulsa is an older Hotel that...


In [62]:
# User text profiles: most recent MAX_PROFILE_REVIEWS training reviews per user                                                                                                                                                   
# Format: [Hotel Name — Rating/5] review text, separated by dividers
BATCH_USERS = 50_000
out_path = os.path.join(OUTPUT_DIR, "user", "user_profiles.parquet")                                                                                                                                                                                                        
                                                                                                                                                                                                                                                                          
n_users = con.execute("SELECT COUNT(*) FROM user_map").fetchone()[0]                                                                                                                                                                                                        
writer  = None                                                                                                                                                                                                                                                              
n_prof  = 0                                                                                                                                                                                                                                                                 
                   
for start in range(0, n_users, BATCH_USERS):
  end = start + BATCH_USERS - 1
  batch = con.execute(f"""
  WITH ranked AS (
      SELECT
          f.user_idx, f.user_id,
          m.hotel_name, f.rating, f.title, f.review_text, f.timestamp,
          ROW_NUMBER() OVER (                                                                                                                                                                                                                                             
              PARTITION BY f.user_id
              ORDER BY f.timestamp DESC                                                                                                                                                                                                                                   
          ) AS rn  
      FROM final_data f
      LEFT JOIN item_metadata m USING (item_idx)                                                                                                                                                                                                                          
      WHERE f.split = 'train'
        AND f.user_idx BETWEEN {start} AND {end}                                                                                                                                                                                                                          
        AND f.review_text IS NOT NULL
        AND LENGTH(TRIM(f.review_text)) > 20                                                                                                                                                                                                                              
  ),
  top AS (SELECT * FROM ranked WHERE rn <= {MAX_PROFILE_REVIEWS})                                                                                                                                                                                                         
  SELECT           
      user_idx, user_id,
      COUNT(*) AS n_profile_reviews,                                                                                                                                                                                                                                      
      STRING_AGG(
          '[' || COALESCE(hotel_name, 'Unknown Hotel')                                                                                                                                                                                                                    
          || ' — ' || CAST(CAST(rating AS INTEGER) AS VARCHAR) || '/5]'
          || CASE WHEN title IS NOT NULL THEN '\n' || title ELSE '' END                                                                                                                                                                                                   
          || '\n' || review_text,
          '\n\n---\n\n'                                                                                                                                                                                                                                                   
          ORDER BY timestamp DESC
      ) AS user_review_history                                                                                                                                                                                                                                            
  FROM top
  GROUP BY user_idx, user_id                                                                                                                                                                                                                                              
  ORDER BY user_idx
  """).fetch_arrow_table()

  if batch.num_rows > 0:
      if writer is None:
          writer = pq.ParquetWriter(out_path, batch.schema)                                                                                                                                                                                                               
      writer.write_table(batch)
      n_prof += batch.num_rows                                                                                                                                                                                                                                            
                   
  print(f"  users {start:>8,}–{end:>8,} | profiles so far: {n_prof:,}", end="\r")                                                                                                                                                                                         

if writer:                                                                                                                                                                                                                                                                  
  writer.close()   

print(f"\nUser profiles built: {n_prof:,} users have training review text")                                                                                                                                                                                                 

#Also in the next cell (the export cell), remove the user_profiles COPY line since this version writes directly to parquet. Change it to:                                                                                                                                    
                   
con.execute(f"COPY user_metadata  TO '{OUT_SAFE}/user/user_metadata.parquet'  (FORMAT PARQUET)")                                                                                                                                                                            
con.execute(f"COPY user_sequences TO '{OUT_SAFE}/user/user_sequences.parquet' (FORMAT PARQUET)")                                                                                                                                                                            
# user_profiles already written to parquet during the batched build above                                                                                                                                                                                                   
print("user/ exports complete.")     

/var/folders/z9/qf_jqt0s51n7dd3yz7l8gh840000gn/T/ipykernel_19417/896022438.py:43: DeprecationWarning: fetch_arrow_table() is deprecated, use to_arrow_table() instead.
  """).fetch_arrow_table()


  users        0–  49,999 | profiles so far: 46,660
User profiles built: 46,660 users have training review text
user/ exports complete.


In [ ]:
# con.execute(f"COPY user_metadata  TO '{OUT_SAFE}/user/user_metadata.parquet'  (FORMAT PARQUET)")
# con.execute(f"COPY user_sequences TO '{OUT_SAFE}/user/user_sequences.parquet' (FORMAT PARQUET)")
# con.execute(f"COPY user_profiles  TO '{OUT_SAFE}/user/user_profiles.parquet'  (FORMAT PARQUET)")
# print("user/ exports complete.")

## Stage 10 — Negative Samples

Two files:
1. **`user_seen_items.parquet`** — for each user, the array of item_idx values seen in
   training. Use this at runtime to filter out already-seen items from any candidate set.
2. **`negative_samples.parquet`** — `NEG_SAMPLES` pre-sampled random negative items per
   user (items from the train item pool that the user has NOT interacted with). Used for
   contrastive fine-tuning (BPR loss, InfoNCE, pairwise ranking objectives).

Sampling strategy: uniform random without replacement from the train item pool, rejecting
items in the user's seen set. Because the matrix is very sparse (~8 interactions per user
against 300K+ items), the rejection rate is negligible (<0.003%).


In [63]:
# Export user seen-item arrays (compact, for runtime filtering)
BATCH_USERS = 50_000                                                                                                                                                                                                                                                        
out_path = os.path.join(OUTPUT_DIR, "training", "user_seen_items.parquet")
                                                                                                                                                                                                                                                                          
n_users = con.execute("SELECT COUNT(*) FROM user_map").fetchone()[0]                                                                                                                                                                                                        
writer  = None                                                                                                                                                                                                                                                              
n_done  = 0                                                                                                                                                                                                                                                                 
              
for start in range(0, n_users, BATCH_USERS):                                                                                                                                                                                                                                
  end = start + BATCH_USERS - 1
  batch = con.execute(f"""                                                                                                                                                                                                                                                
  SELECT      
      user_idx, user_id,                                                                                                                                                                                                                                                  
      LIST(item_idx ORDER BY timestamp ASC) AS seen_item_idxs,
      LIST(item_id  ORDER BY timestamp ASC) AS seen_item_ids                                                                                                                                                                                                              
  FROM final_data
  WHERE split = 'train'                                                                                                                                                                                                                                                   
    AND user_idx BETWEEN {start} AND {end}
  GROUP BY user_idx, user_id                                                                                                                                                                                                                                              
  ORDER BY user_idx
  """).fetch_arrow_table()
                                                                                                                                                                                                                                                                          
  if batch.num_rows > 0:
      if writer is None:                                                                                                                                                                                                                                                  
          writer = pq.ParquetWriter(out_path, batch.schema)
      writer.write_table(batch)
      n_done += batch.num_rows                                                                                                                                                                                                                                            

  print(f"  users {start:>8,}–{end:>8,} | written: {n_done:,}", end="\r")                                                                                                                                                                                                 
              
if writer:                                                                                                                                                                                                                                                                  
  writer.close()

print(f"\nuser_seen_items exported: {n_done:,} users")      

/var/folders/z9/qf_jqt0s51n7dd3yz7l8gh840000gn/T/ipykernel_19417/1108334463.py:21: DeprecationWarning: fetch_arrow_table() is deprecated, use to_arrow_table() instead.
  """).fetch_arrow_table()


  users        0–  49,999 | written: 46,660
user_seen_items exported: 46,660 users


In [64]:
# Pre-sample NEG_SAMPLES random negatives per user using numpy
# Runs in chunks of 50K users to keep memory usage modest

np.random.seed(42)                                                                                                                                                                                                                                                          
CHUNK         = 100_000
out_path      = os.path.join(OUTPUT_DIR, "training", "negative_samples.parquet")                                                                                                                                                                                            
schema        = pa.schema([("user_idx", pa.int32()), ("neg_item_idx", pa.int32())])
                                                                                                                                                                                                                                                                          
# All item indices in the train pool                                                                                                                                                                                                                                        
all_items = con.execute(                                                                                                                                                                                                                                                    
  "SELECT DISTINCT item_idx FROM final_data WHERE split='train' ORDER BY item_idx"                                                                                                                                                                                        
).fetchdf()["item_idx"].to_numpy(dtype=np.int32)                                                                                                                                                                                                                            
                                                                                                                                                                                                                                                                          
# User seen-item sets                                                                                                                                                                                                                                                       
user_seen_df = pd.read_parquet(os.path.join(OUTPUT_DIR, "training", "user_seen_items.parquet"),                                                                                                                                                                             
                             columns=["user_idx", "seen_item_idxs"])                                                                                                                                                                                                      

n_items_total = len(all_items)                                                                                                                                                                                                                                              
writer        = None

print(f"Sampling {NEG_SAMPLES} negatives for {len(user_seen_df):,} users ...")                                                                                                                                                                                              

for chunk_start in range(0, len(user_seen_df), CHUNK):                                                                                                                                                                                                                      
  chunk      = user_seen_df.iloc[chunk_start : chunk_start + CHUNK]
  chunk_size = len(chunk)                                                                                                                                                                                                                                                 

  rand_items = all_items[                                                                                                                                                                                                                                                 
      np.random.randint(0, n_items_total, size=(chunk_size, NEG_SAMPLES * 4))
  ]                                                                                                                                                                                                                                                                       

  u_idxs, n_idxs = [], []                                                                                                                                                                                                                                                 
  for i, row in enumerate(chunk.itertuples(index=False)):
      seen_set = set(row.seen_item_idxs)
      negs = [int(c) for c in rand_items[i] if c not in seen_set][:NEG_SAMPLES]                                                                                                                                                                                           

      if len(negs) < NEG_SAMPLES:                                                                                                                                                                                                                                         
          extra = all_items[np.random.randint(0, n_items_total, size=NEG_SAMPLES * 20)]
          seen_and_neg = seen_set | set(negs)                                                                                                                                                                                                                             
          negs += [int(c) for c in extra if c not in seen_and_neg]                                                                                                                                                                                                        
          negs = negs[:NEG_SAMPLES]
                                                                                                                                                                                                                                                                          
      u_idxs.extend([row.user_idx] * len(negs))                                                                                                                                                                                                                           
      n_idxs.extend(negs)
                                                                                                                                                                                                                                                                          
  table = pa.table({"user_idx": pa.array(u_idxs, pa.int32()),
                    "neg_item_idx": pa.array(n_idxs, pa.int32())})
  if writer is None:                                                                                                                                                                                                                                                      
      writer = pq.ParquetWriter(out_path, schema)
  writer.write_table(table)                                                                                                                                                                                                                                               
              
  done = min(chunk_start + CHUNK, len(user_seen_df))                                                                                                                                                                                                                      
  print(f"  {done:>10,} / {len(user_seen_df):,} users processed", end="\r")
                                                                                                                                                                                                                                                                          
if writer:      
  writer.close()
print(f"\nNegatives saved: {out_path}") 

Sampling 100 negatives for 46,660 users ...
      46,660 / 46,660 users processed
Negatives saved: /Users/jaswanthimandalapu/Documents/Sem2/CMPE 256 (Recommender Systems)/Final Project/llm_rec_data_20interactions/training/negative_samples.parquet


## Stage 11 — Final Summary


In [65]:
# Dataset statistics
stats = con.execute("""
SELECT split,
    COUNT(*)             AS n_interactions,
    COUNT(DISTINCT user_id) AS n_users,
    COUNT(DISTINCT item_id) AS n_items,
    AVG(rating)          AS mean_rating,
    COUNT(review_text)   AS n_with_text,
    ROUND(100.0 * COUNT(review_text) / COUNT(*), 1) AS pct_with_text
FROM final_data
GROUP BY split
ORDER BY split DESC
""").fetchdf()

print("=" * 70)
print("DATASET STATISTICS")
print("=" * 70)
display(stats)

# File listing
print("\n" + "=" * 70)
print("EXPORTED FILES")
print("=" * 70)
total_mb = 0
for root, dirs, files in os.walk(OUTPUT_DIR):
    dirs[:] = sorted(d for d in dirs if d != "tmp")
    for fname in sorted(files):
        if fname.endswith((".parquet", ".json")):
            fpath = os.path.join(root, fname)
            mb = os.path.getsize(fpath) / 1e6
            total_mb += mb
            rel = os.path.relpath(fpath, OUTPUT_DIR)
            print(f"  {rel:<55s}  {mb:8.1f} MB")
print(f"\n  Total: {total_mb:.1f} MB")

# Save pipeline config for reproducibility
n_train = con.execute("SELECT COUNT(*) FROM final_data WHERE split='train'").fetchone()[0]
n_valid = con.execute("SELECT COUNT(*) FROM final_data WHERE split='valid'").fetchone()[0]
n_test  = con.execute("SELECT COUNT(*) FROM final_data WHERE split='test'").fetchone()[0]
n_users = con.execute("SELECT COUNT(DISTINCT user_idx) FROM final_data").fetchone()[0]
n_items = con.execute("SELECT COUNT(DISTINCT item_idx) FROM final_data").fetchone()[0]

config = {
    "pipeline": "llm_rec_pipeline",
    "source_file": RAW_FILE,
    "k_core": K_CORE,
    "split": "leave_last_out_temporal",
    "max_profile_reviews": MAX_PROFILE_REVIEWS,
    "neg_samples_per_user": NEG_SAMPLES,
    "global_mean_train": round(global_mean, 6),
    "bayesian_smoothing_m": m_smooth,
    "n_users": n_users,
    "n_items": n_items,
    "n_train": n_train,
    "n_valid": n_valid,
    "n_test":  n_test,
}

with open(os.path.join(OUTPUT_DIR, "pipeline_config.json"), "w") as f:
    json.dump(config, f, indent=2)

print("\npipeline_config.json saved.")
print("\nPipeline complete. All data needed for LLM-based recommendation is ready.")

DATASET STATISTICS


,split,n_interactions,n_users,n_items,mean_rating,n_with_text,pct_with_text
0,valid,46660,46660,18731,4.143892,46660,100.0
1,train,1306782,46660,27197,4.044277,1306782,100.0
2,test,46660,46660,18355,4.221496,46660,100.0



EXPORTED FILES
  core/duplicate_features.parquet                              69.8 MB
  core/interactions_test.parquet                               27.1 MB
  core/interactions_train.parquet                             744.4 MB
  core/interactions_valid.parquet                              26.9 MB
  core/item_id_mapping.parquet                                  1.0 MB
  core/user_id_mapping.parquet                                  0.6 MB
  item/item_metadata.parquet                                    2.8 MB
  item/item_profiles.parquet                                  124.9 MB
  training/negative_samples.parquet                             9.7 MB
  training/user_seen_items.parquet                             61.8 MB
  user/user_metadata.parquet                                    1.4 MB
  user/user_profiles.parquet                                  244.4 MB
  user/user_sequences.parquet                                 750.8 MB

  Total: 2065.7 MB

pipeline_config.json saved.

Pipeline co

In [29]:
file_path = "HotelRec.txt"

with open(file_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 50:
            break
        print(line.rstrip())

{"hotel_url": "Hotel_Review-g194775-d1121769-Reviews-Hotel_Baltic-Giulianova_Province_of_Teramo_Abruzzo.html", "author": "violettaf340", "date": "2019-01-01T00:00:00", "rating": 5.0, "title": "Xmas holiday", "text": "We went here with our kids for Xmas holiday and we really liked it. Large options of food for breakfast and lunch , you can really taste the quality of the food in there. The surrounding area is nice and clean. Good experience. Hardly recommended .", "property_dict": {}}
{"hotel_url": "Hotel_Review-g194775-d1121769-Reviews-Hotel_Baltic-Giulianova_Province_of_Teramo_Abruzzo.html", "author": "Lagaiuzza", "date": "2016-01-01T00:00:00", "rating": 5.0, "title": "Baltic, what else?", "text": "We have spent in this hotel our summer holidays both in summer 2014 and 2015- I was with my husband and my child ( 4 years old at present). I do really recommend this place- Staff si high qualified, Kind and really helpful- Animation staff get You involved, but always with discrection - Min